In [5]:
import numpy as np
import pynucastro as pyna
import pandas as pd
from scipy.constants import physical_constants
m_u=physical_constants['atomic mass constant energy equivalent in MeV'][0]
m_e=physical_constants['electron mass energy equivalent in MeV'][0]
from scipy.constants import speed_of_light as c

In [6]:
nuclear_data=pd.read_csv('Nuclear_data\Mass\winvne_v2.0.dat',skiprows=lambda x: not (x>7854 and (x-7855)%4==0),delim_whitespace=True,names=['name','A','Z','N','spin','Mass excess (Mev)','source'])
def index_winv_v2(z,n):
        
    idx_start = np.searchsorted(nuclear_data['Z'], z, side='left')
    idx_end = np.searchsorted(nuclear_data['Z'], z, side='right')
    if idx_end>idx_start:
        j=np.searchsorted(nuclear_data['N'][idx_start:idx_end],n)+idx_start
        if nuclear_data['N'][j]==n and nuclear_data['Z'][j]==z:
            return j
        else:
            return 'None'
    else:
        return 'None'
    
def parameters(parityZ,parityN):
    if parityZ==0 and parityN==0:#even-even
        return [0.05, 13.30921, -0.30203, 0.00234, -6.89e-06]
    elif parityZ==0 and parityN==1:#even-odd
        return [0.05, 13.13574, -0.25674, 0.0017, -4.44e-06]
    elif parityZ==1 and parityN==1:#odd-odd
        return [0.03, 8.69628, -0.11596, 2.90e-04, 1.25e-08]
    elif parityZ==1 and parityN==0:#odd-even
        return [0.03, 10.28474, -0.10789, -1.28e-05, 1.32e-06]

In [ ]:
decays=[]
for i in range(1,43):
    if i%2==0:
        for j in range(int(i),int(i*3)+1):
            if i+j>3 and i+j<118:
                if index_winv_v2(i+1,j-1)!='None' and index_winv_v2(i,j)!='None':
                    mp=nuclear_data['Mass excess (Mev)'][index_winv_v2(i,j)]+(i+j)*m_u
                    md=nuclear_data['Mass excess (Mev)'][index_winv_v2(i+1,j-1)]+(i+j)*m_u
                    Q_value=(mp-md)
                    if Q_value>0:
                        lp=parameters(0,j%2)
                        log_T=0
                        for k in range(4):
                            log_T+=lp[k+1]*(((i+1)**(lp[0]))*np.sqrt(Q_value*1000))**k
                        decays.append([nuclear_data['name'][index_winv_v2(i,j)],i,j,np.log(2)/(10**log_T),nuclear_data['name'][index_winv_v2(i+1,j-1)],Q_value])
                        
                    
    else:
        for j in range(int(i),int(i*3)+1):
            if i+j>3 and i+j< 118:
                if index_winv_v2(i+1,j-1)!='None' and index_winv_v2(i,j)!='None':
                    mp=nuclear_data['Mass excess (Mev)'][index_winv_v2(i,j)]+(i+j)*m_u
                    md=nuclear_data['Mass excess (Mev)'][index_winv_v2(i+1,j-1)]+(i+j)*m_u
                    Q_value=(mp-md)
                    if Q_value>0:
                        lp=parameters(1,j%2)
                        log_T=0
                        for k in range(4):
                            log_T+=lp[k+1]*(((i+1)**(lp[0]))*np.sqrt(Q_value*1000))**k
                        decays.append([nuclear_data['name'][index_winv_v2(i,j)],i,j,np.log(2)/(10**log_T),nuclear_data['name'][index_winv_v2(i+1,j-1)],Q_value])
                        

dats=pd.DataFrame(decays)
dats.to_csv('Nuclear_data/decays/Empirical_beta_decay_example.csv',index=False,header=['name_p','Z','N','decay rate (1/s)','name_d','Q_value'])
